In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

machine_numbers = ["M1201", "M1202", "M1203", "M1204", "M1205"]
locations = ["P1", "P2", "P3", "P4"]

# Daily readings from January 1, 2024 through December 31, 2025
timestamps = pd.date_range(
    start="2024-01-01",
    end="2025-12-31",
    freq="D"
)

# Each machine, day, and location is one bearing-location reading
point_day_keys = [
    (machine, day_number, location)
    for machine in machine_numbers
    for day_number in range(len(timestamps))
    for location in locations
]

# Select 5% of bearing-location readings as high-vibration cases
high_count = round(0.05 * len(point_day_keys))
selected_indices = rng.choice(
    len(point_day_keys),
    size=high_count,
    replace=False
)
high_vibration_keys = {
    point_day_keys[i] for i in selected_indices
}

records = []

for machine in machine_numbers:
    low_vibration_level = {
        location: rng.uniform(4.0, 5.6)
        for location in locations
    }

    temperature_level = {
        location: rng.uniform(40.0, 54.0)
        for location in locations
    }

    for day_number, timestamp in enumerate(timestamps):
        row = {
            "Machine_number": machine,
            "time stamp": timestamp
        }

        # Seasonal temperature pattern, repeating approximately once per year
        seasonal_change = 2.5 * np.sin(
            2 * np.pi * day_number / 365.25
        )

        for location in locations:
            key = (machine, day_number, location)

            if key in high_vibration_keys:
                # All three directions exceed 7.0 mm/s
                vibration_center = rng.uniform(7.8, 9.9)
                offsets = rng.uniform(-0.55, 0.55, size=3)
            else:
                # Keep these readings below 6.5 mm/s
                low_vibration_level[location] += rng.normal(0, 0.03)
                low_vibration_level[location] = np.clip(
                    low_vibration_level[location], 3.65, 5.85
                )
                vibration_center = low_vibration_level[location]
                offsets = rng.uniform(-0.55, 0.55, size=3)

            row[f"{location}-V"] = round(
                float(vibration_center + offsets[0]), 2
            )
            row[f"{location}-H"] = round(
                float(vibration_center + offsets[1]), 2
            )
            row[f"{location}-A"] = round(
                float(vibration_center + offsets[2]), 2
            )

            temperature = (
                temperature_level[location]
                + seasonal_change
                + rng.normal(0, 1.2)
            )
            temperature = np.clip(temperature, 30.0, 65.0)
            row[f"{location}-T"] = round(float(temperature), 2)

        records.append(row)

df = pd.DataFrame(records)

# Arrange columns in the requested order
columns = ["Machine_number", "time stamp"]
for location in locations:
    columns.extend([
        f"{location}-V",
        f"{location}-H",
        f"{location}-A",
        f"{location}-T"
    ])

df = df[columns]

# Add about 8% missing values across all columns, including ID and timestamp
missing_count = round(df.size * 0.08)
missing_indices = rng.choice(df.size, size=missing_count, replace=False)

for cell_index in missing_indices:
    row_index, column_index = divmod(cell_index, df.shape[1])
    df.iat[row_index, column_index] = np.nan

output_file = "pump_motor_vibration_2024_2025_daily.csv"
df.to_csv(output_file, index=False)

# Report the percentage of non-missing vibration readings above 7.0
vibration_columns = [
    f"{location}-{direction}"
    for location in locations
    for direction in ["V", "H", "A"]
]

observed_vibrations = df[vibration_columns].stack()
high_percentage = (observed_vibrations > 7.0).mean()

print(f"Saved dataset to: {output_file}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Missing cells: {df.isna().sum().sum()} ({df.isna().sum().sum() / df.size:.2%})")
print(f"Non-missing vibration readings above 7.0: {high_percentage:.2%}")
print(df.head())

Saved dataset to: pump_motor_vibration_2024_2025_daily.csv
Rows: 3655
Columns: 18
Missing cells: 5263 (8.00%)
Non-missing vibration readings above 7.0: 5.00%
  Machine_number time stamp  P1-V  P1-H  P1-A   P1-T  P2-V  P2-H  P2-A   P2-T  \
0          M1201 2024-01-01  4.85  5.71  5.17  50.53  5.18  5.60  5.43  44.80   
1          M1201 2024-01-02  5.19  5.60  5.01  48.80  4.88  5.55  5.40  42.14   
2          M1201 2024-01-03  5.54  4.90  5.68  51.39  4.54  4.59  5.20  42.32   
3          M1201 2024-01-04  5.42  5.66   NaN  49.03   NaN  5.29  5.59  42.26   
4          M1201 2024-01-05  5.44  5.37   NaN  52.43  5.14  5.27  5.21  45.30   

   P3-V  P3-H  P3-A   P3-T  P4-V  P4-H  P4-A   P4-T  
0  5.07  4.77  5.03  45.76  5.66  5.09  5.60  43.85  
1  4.88   NaN  4.84  45.98  5.06  5.95  4.99  43.60  
2  4.80  3.93  4.75  44.84  5.74  5.89  5.01  43.56  
3  4.56   NaN  4.85  43.03  5.10  5.20  6.07  43.75  
4  4.45  4.34  4.42  47.76  5.30   NaN  5.47  44.97  
